# Business Forecasting · Lecture 2 | Time-series adjustments and decomposition
### Python tutorial — original textbook data · Google Colab

**How to use:** Run from top to bottom. When asked, upload **one ZIP containing the four CSVs** or select **the four CSV files individually**. Explanations tell you what to do, what the Python operations mean, and how to read each result. There are no online data downloads or invented observations.

**Data:** `global_economy.csv`, `aus_retail.csv`, `us_employment.csv`, `aus_production.csv` — original CSVs supplied with the lecture. The notebook works independently of the PowerPoint.

## 1 · Import the tools and upload the data
We need `pandas` for tables and dates, `numpy` for transformations, `matplotlib` for graphs, and `statsmodels` for decomposition. Run this setup once.

In [ ]:
from pathlib import Path                   # File and folder paths.
import io                                  # Lets Python read an uploaded ZIP from memory.
import zipfile                             # Opens ZIP archives.
import numpy as np                         # Square roots, logarithms, numerical checks.
import pandas as pd                        # CSVs, grouping, dates and moving averages.
import matplotlib.pyplot as plt            # Graphs.
from statsmodels.tsa.seasonal import STL, seasonal_decompose  # Two decomposition methods.
from IPython.display import display        # Display tables inside Colab.

plt.rcParams['figure.figsize'] = (11, 5)    # Comfortable default chart size.
plt.rcParams['axes.grid'] = True           # Make charts easier to read.
DATA_DIR = Path('Lecture2_Data')    # Our notebook's data folder.
DATA_DIR.mkdir(exist_ok=True)              # Create it if necessary.
REQUIRED = ['global_economy.csv', 'aus_retail.csv',
            'us_employment.csv', 'aus_production.csv']

### Upload ONE ZIP, or select the four CSVs together
In Google Colab, running the next cell opens a file picker **if the data are not already loaded**. You can upload the supplied ZIP even if it contains a folder inside. Or upload the four CSVs directly (multi-select). The notebook extracts only the four required original files. **When reconnecting to a new Colab runtime, upload again.**

In [ ]:
# Show the upload dialog only when one or more required files are absent.
if not all((DATA_DIR / name).exists() for name in REQUIRED):
    try:
        from google.colab import files     # Colab's browser upload dialog.
        uploaded = files.upload()          # Returns {filename: file bytes} for selected files.
    except ImportError:
        uploaded = {}                      # In ordinary Jupyter, put CSVs in DATA_DIR yourself.

    for filename, contents in uploaded.items():
        if filename.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(contents)) as archive:
                for member in archive.namelist():
                    short_name = Path(member).name  # Ignore folders inside the ZIP.
                    if short_name in REQUIRED:      # Only take lecture datasets.
                        (DATA_DIR / short_name).write_bytes(archive.read(member))
        elif Path(filename).name in REQUIRED:
            (DATA_DIR / Path(filename).name).write_bytes(contents)  # Save an individual CSV.

missing = [name for name in REQUIRED if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError('Upload the original ZIP or missing CSVs: ' + ', '.join(missing))
print('Ready: all four original book CSVs are available.')

### Open the four datasets
`pd.read_csv()` loads a CSV as a **DataFrame**, meaning a table with rows and named columns. `parse_dates=[...]` makes the specified column a real date so we can sort and plot time correctly.

- **`global_economy`** — annual records for multiple countries; `unique_id` = country, `ds` = year, `GDP`, `Population`, `CPI`, `Exports` are measures.
- **`aus_retail`** — monthly retail turnover by Australian state and industry; `Month` is the date.
- **`us_employment`** — monthly US industry employment; `unique_id` = industry and `y` = employed persons **in thousands**.
- **`aus_production`** — quarterly Australian production; `Beer` is the series used below.

In [ ]:
global_economy = pd.read_csv(DATA_DIR / 'global_economy.csv')
aus_retail = pd.read_csv(DATA_DIR / 'aus_retail.csv', parse_dates=['Month'])
us_employment = pd.read_csv(DATA_DIR / 'us_employment.csv', parse_dates=['ds'])
aus_production = pd.read_csv(DATA_DIR / 'aus_production.csv', parse_dates=['ds'])
print('Book datasets loaded.')

---
# Part A · Adjusting historical data
The purpose of adjustment is to separate changes due to the **calendar, population or price level** from changes in the underlying business measure. Adjustment changes the *meaning* of the series: read the vertical-axis label each time.

## 2 · Calendar adjustment: Australian food retailing
**Question:** Does a larger monthly total mean that stores sold more *per day*? Not necessarily: February has fewer calendar days than March.

From `aus_retail.csv`, choose the industry **Food retailing**, add turnover across states for each month, then divide by the actual number of days in that month. In the code, `.groupby('Month')` puts rows from the same month together; `.sum()` adds their turnover; `.dt.days_in_month` supplies 28, 29, 30 or 31 as appropriate.

In [ ]:
food_rows = aus_retail.query("Industry == 'Food retailing'").copy()  # Keep only Food retailing; copy makes our own working table.
food_monthly = (food_rows.groupby('Month', as_index=False)['Turnover']
                .sum().sort_values('Month'))                  # One national turnover total per month, ordered by date.
food_monthly['days'] = food_monthly['Month'].dt.days_in_month  # Calendar days for each observed month.
food_monthly['per_day'] = food_monthly['Turnover'] / food_monthly['days']  # Turnover per calendar day.
display(food_monthly.head(6))

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].plot(food_monthly['Month'], food_monthly['Turnover'])
ax[0].set_title('Australian food retailing: monthly total')
ax[0].set_ylabel('Monthly turnover (source units)')
ax[1].plot(food_monthly['Month'], food_monthly['per_day'])
ax[1].set_title('Same turnover divided by calendar days')
ax[1].set_ylabel('Turnover per calendar day')
ax[1].set_xlabel('Month')
plt.tight_layout(); plt.show()

**Read the output:** Both graphs describe the **same retail observations**, but their units differ. Month-to-month differences caused simply by month length are reduced in the second graph. This is a *calendar-day* adjustment; a business with different weekday opening hours might instead need a *trading-day* adjustment.

## 3 · Population adjustment: Australian GDP per capita
**Question:** Is total GDP enough to compare economic output over time when the population changes? We also want **GDP per person = GDP ÷ Population**.

The economy CSV contains many countries. `query("unique_id == 'Australia'")` **selects only rows where the country equals Australia**. The `==` asks whether two values are equal; the quote marks keep `Australia` as text. `.copy()` makes an independent working table, so creating new columns will not change `global_economy`. `.sort_values('ds')` puts years in order.

In [ ]:
australia = (global_economy.query("unique_id == 'Australia'")
             .copy().sort_values('ds'))  # Australia only; keep a separate table in year order.
australia['GDP_per_person'] = australia['GDP'] / australia['Population']  # Adjust total GDP for population.
display(australia[['ds', 'GDP', 'Population', 'GDP_per_person']].head())

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].plot(australia['ds'], australia['GDP'])
ax[0].set_title('Australia: total GDP'); ax[0].set_ylabel('GDP (source units)')
ax[1].plot(australia['ds'], australia['GDP_per_person'])
ax[1].set_title('Australia: GDP per person'); ax[1].set_ylabel('GDP per person')
ax[1].set_xlabel('Year')
plt.tight_layout(); plt.show()

**Read the output:** The first graph answers “how large is the economy?” The second answers “how much output corresponds to each person?” The two curves need not grow at the same rate. This is a **population adjustment**, not an inflation adjustment: GDP per person here is still in the price basis reported by the source data.

## 4 · Inflation adjustment: newspaper and book retailing
**Question:** Has turnover really increased, or are prices simply higher? Compare **nominal turnover** (money reported in each year) with **turnover expressed in 2010 prices**.

We use monthly `Newspaper and book retailing` turnover from `aus_retail.csv` and annual Australian `CPI` from `global_economy.csv`. First combine states and months into an annual turnover series. `merge(..., on='year')` attaches the CPI for that **same year**. Then:

**Turnover in 2010 prices = annual turnover × CPI in 2010 ÷ CPI in that year.**

In [ ]:
print_retail = aus_retail.query("Industry == 'Newspaper and book retailing'").copy()  # Choose the book's print-retail industry.
print_retail['year'] = print_retail['Month'].dt.year      # Turn each monthly date into its calendar year.
annual_print = print_retail.groupby('year', as_index=False)['Turnover'].sum()  # Total all reported months and states in each year.
cpi = australia[['ds', 'CPI']].rename(columns={'ds': 'year'})   # Give both tables the same year-column name.
print_prices = annual_print.merge(cpi, on='year', how='inner')  # Keep years found in both turnover and CPI.
cpi_2010 = cpi.loc[cpi['year'] == 2010, 'CPI'].iloc[0]     # Select the CPI value for reference year 2010.
print_prices['in_2010_prices'] = print_prices['Turnover'] * cpi_2010 / print_prices['CPI']  # Remove CPI change.
display(print_prices[['year', 'Turnover', 'CPI', 'in_2010_prices']].head())

plt.plot(print_prices['year'], print_prices['Turnover'], label='Nominal turnover')
plt.plot(print_prices['year'], print_prices['in_2010_prices'], label='Turnover in 2010 prices')
plt.title('Australian newspaper and book retailing: inflation adjustment')
plt.xlabel('Year'); plt.ylabel('Annual turnover (source units)'); plt.legend()
plt.tight_layout(); plt.show()

**Read the output:** In 2010 both turnover measures coincide: `CPI_2010 / CPI_2010 = 1`. A nominal series can look stable or rising even when inflation-adjusted activity is falling. CPI adjustment changes the **price basis**, not the number of newspaper or book units sold.

---
# Part B · Mathematical transformations
If swings get larger as a series grows, we may change its mathematical scale to make patterns easier to compare. A transformation does **not** change the underlying observations; it changes how their size is represented. For positive values, common choices are square root, cube root, natural log and negative reciprocal.

## 5 · Australian food turnover: original and transformed scales
Use the same national monthly Food retailing series created in Section 2. Compare **the same dates** on different vertical scales. `np.sqrt(x)` takes the square root; `np.cbrt(x)` the cube root; `np.log(x)` the **natural logarithm**; `-1/x` is a strong inverse transformation. Larger values are compressed by different amounts.

In [ ]:
food = food_monthly[['Month', 'Turnover']].copy()  # Start again with original monthly turnover, not turnover per day.
food['sqrt'] = np.sqrt(food['Turnover'])   # Square-root scale.
food['cbrt'] = np.cbrt(food['Turnover'])   # Cube-root scale.
food['log'] = np.log(food['Turnover'])     # Natural-log scale (positive observations only).
food['negative_inverse'] = -1 / food['Turnover']  # Negative reciprocal, also needs nonzero values.

fig, ax = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
for axis, column, title in zip(ax.flat,
    ['Turnover', 'sqrt', 'cbrt', 'log', 'negative_inverse'],
    ['Original', 'Square root', 'Cube root', 'Natural log', 'Negative inverse']):
    axis.plot(food['Month'], food[column])   # Same time series, different vertical transformation.
    axis.set_title(title); axis.set_ylabel('Transformed scale')
ax.flat[-1].axis('off')                      # Only five versions: leave the sixth panel empty.
fig.suptitle('Australian food retailing: compare transformations')
plt.tight_layout(); plt.show()

**Read the output:** Compare how peak-to-trough swings change between the beginning and end of each graph. The log is especially useful when **relative or percentage changes** matter; negative inverse can compress values too aggressively for easy interpretation. Never compare the raw *heights* of two transformed graphs as though they used the same unit.

One useful log fact: `log(this month) − log(last month)` is approximately the proportional change when changes are small. For the **exact** percentage change use the next calculation.

In [ ]:
food['log_change'] = food['log'].diff()                 # Current log minus previous month's log.
food['percent_change_from_log'] = 100 * (np.exp(food['log_change']) - 1)  # Convert back to the exact percent change.
display(food[['Month', 'Turnover', 'log_change', 'percent_change_from_log']].head(6))

---
# Part C · Patterns, decomposition and seasonal adjustment
A time series may include **trend-cycle** (longer-run movement), **seasonality** (a recurring calendar pattern), and **remainder** (variation not captured by the other two). STL estimates these three parts using locally fitted smoothing curves (LOESS).

In an **additive** decomposition: **Observed = Trend + Seasonal + Remainder**. Seasonal values are *amounts* above or below trend; they are not seasonal percentages.

## 6 · US Retail Trade employment: select the correct series
`us_employment.csv` includes multiple industries. Here `unique_id` names the industry, `ds` is the month, and `y` is employment **in thousands of people**. We keep `Retail Trade` from January 1990 onward.

`query("unique_id == 'Retail Trade' and ds >= '1990-01-01'")` has **two conditions**: the industry must be Retail Trade **and** the date must be on or after January 1990. `.copy()` again keeps a separate working dataset; `.reset_index(drop=True)` gives the selected rows simple consecutive row numbers.

In [ ]:
employment = (us_employment.query("unique_id == 'Retail Trade' and ds >= '1990-01-01'")
              .copy().sort_values('ds').reset_index(drop=True))  # One industry, date range and proper month order.
plt.plot(employment['ds'], employment['y'])             # Plot the ORIGINAL monthly employment first.
plt.title('US Retail Trade employment: observed data')
plt.xlabel('Month'); plt.ylabel('Employed persons (thousands)')
plt.tight_layout(); plt.show()

**Read the output:** Look for longer-run increases or declines, recurring increases at similar times each year, and irregular disruptions. We will now separate these features, rather than asking one line to explain all of them at once.

## 7 · STL: break employment into three components
`STL(series, period=12, robust=True).fit()` performs the decomposition. Here `period=12` tells Python there are **12 monthly observations in one yearly seasonal cycle**; it is not a request to forecast 12 months. `robust=True` reduces the impact of unusual observations when the smoother is estimated. `.fit()` actually calculates the components.

In [ ]:
stl = STL(employment['y'], period=12, robust=True).fit()  # Fit monthly additive STL to the selected series.
components = employment[['ds', 'y']].copy()             # Keep the dates and original employment.
components['trend'] = stl.trend                         # Smooth longer-run level of employment.
components['seasonal'] = stl.seasonal                   # Month-specific above/below-trend contribution.
components['remainder'] = stl.resid                     # Observed minus trend minus seasonality.
display(components.head(5))

fig, ax = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for axis, column, title in zip(ax, ['y', 'trend', 'seasonal', 'remainder'],
                               ['Observed', 'Trend-cycle', 'Seasonal', 'Remainder']):
    axis.plot(components['ds'], components[column])
    axis.set_title(title)
ax[-1].set_xlabel('Month')
fig.suptitle('US Retail Trade: STL decomposition (thousands of people)')
plt.tight_layout(); plt.show()

**Read the four panels in order:** Observed is the original monthly number. Trend-cycle shows the slower underlying direction. Seasonal shows how much a given month sits above or below the trend. Remainder is what is not explained by those two estimates. A small remainder is desirable, but STL is **description**, not by itself a complete forecasting model.

## 8 · Trend versus seasonally adjusted employment
A **seasonally adjusted** series is the observed series with the recurring seasonal part removed:

**Seasonally adjusted = Observed − Seasonal = Trend + Remainder.**

It is *not identical* to the smooth trend: the adjustment still contains irregular changes.

In [ ]:
components['adjusted'] = components['y'] - components['seasonal']  # Remove only the STL seasonal contribution.
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].plot(components['ds'], components['y'], label='Observed')
ax[0].plot(components['ds'], components['trend'], label='Trend')
ax[0].set_title('What is the underlying longer-run movement?'); ax[0].legend()
ax[1].plot(components['ds'], components['y'], label='Observed')
ax[1].plot(components['ds'], components['adjusted'], label='Seasonally adjusted')
ax[1].set_title('What remains when regular seasonality is removed?'); ax[1].legend()
ax[1].set_xlabel('Month'); fig.supylabel('Employment (thousands)')
plt.tight_layout(); plt.show()

**Read the output:** The top graph compares raw employment to the smoothed baseline. The bottom graph removes recurring month effects but retains nonseasonal surprises. Analysts use seasonally adjusted employment to compare neighboring months without automatically treating the normal annual hiring cycle as a new change in economic conditions.

## 9 · Seasonal subseries: does a month behave the same across years?
A seasonal subseries gives **one small chart per calendar month**. For example, the December panel shows December's estimated seasonal employment effect across successive years. This exposes changes in seasonal strength that one average monthly index could hide.

`.dt.month` produces numbers 1–12 from dates; `range(1, 13)` loops through all twelve months; `axes.flat[month-1]` chooses the matching panel in the 3-by-4 chart grid.

In [ ]:
subseries = components[['ds', 'seasonal']].copy()
subseries['month'] = subseries['ds'].dt.month       # 1 = Jan, ..., 12 = Dec.
subseries['year'] = subseries['ds'].dt.year         # Compare the same month across years.
fig, axes = plt.subplots(3, 4, figsize=(14, 8), sharex=True, sharey=True)
for month in range(1, 13):
    one_month = subseries.query('month == @month')  # @month takes the Python loop variable into query().
    axis = axes.flat[month - 1]                    # One panel per calendar month.
    axis.plot(one_month['year'], one_month['seasonal'])
    axis.axhline(0, linewidth=0.6, linestyle='--')  # Zero = no effect relative to trend.
    axis.set_title(pd.Timestamp(2000, month, 1).strftime('%b'))
fig.suptitle('US Retail Trade: STL seasonal effect by calendar month')
fig.supxlabel('Year'); fig.supylabel('Effect (thousands of people)')
plt.tight_layout(); plt.show()

**Read the output:** If a month’s line changes through the years, its seasonal effect is not perfectly constant. A December spike is read in **thousands of employees relative to the estimated trend**, not as the industry's total December employment.

---
# Part D · Moving averages and the trend-cycle
A **moving average** replaces an observation with the average of nearby observations. A **centered 5-MA** for a year uses *two years before + that year + two years after*. A larger window gives a smoother curve but loses more values at the ends of the sample. Centered MAs are for historical trend estimation; they use future observations relative to the point being smoothed, so do **not** use them as a real-time forecasting input.

## 10 · Australian exports: compare different moving-average orders
The `Exports` column of Australia's annual `global_economy.csv` rows is **exports as a percentage of GDP**, not the dollar value of exports. In `.rolling(window=5, center=True).mean()`, `window=5` means five consecutive years and `center=True` positions the result at the middle year.

Each MA below is calculated from the **original export series**, not from a previously smoothed MA.

In [ ]:
exports = australia[['ds', 'Exports']].copy().sort_values('ds')
for order in [3, 5, 7, 9, 11, 13, 15]:
    exports[f'ma_{order}'] = exports['Exports'].rolling(window=order, center=True).mean()  # One new column per window size.
display(exports[['ds', 'Exports', 'ma_3', 'ma_5', 'ma_15']].head(10))

plt.plot(exports['ds'], exports['Exports'], label='Observed exports')
for order in [3, 7, 15]:
    plt.plot(exports['ds'], exports[f'ma_{order}'], label=f'{order}-MA')  # Compare short, medium and long windows.
plt.title('Australian exports: effect of MA length')
plt.xlabel('Year'); plt.ylabel('Exports (% of GDP)'); plt.legend()
plt.tight_layout(); plt.show()

**Read the output:** The 3-MA follows the raw line more closely; the 15-MA changes more slowly. Blank values (`NaN`) near the edges are **expected**: a centered average cannot be calculated where earlier or later observations do not exist. A centered 15-MA needs seven years on each side, so its first seven and last seven years have no estimate.

## 11 · Even seasonal length: why we need a 2×4-MA
Australian Beer production in `aus_production.csv` is **quarterly**: four quarters make one yearly season. A plain 4-quarter average has its mathematical center **between quarters**. A **2×4-MA** averages neighboring 4-MAs so the trend estimate lines up with an actual quarter.

`rolling(4, center=True)` makes intermediate four-quarter means. `rolling(2).mean()` averages adjacent intermediate means. `.shift(-1)` moves the result up one row so the final value is aligned with its true center. This yields weights **1, 2, 2, 2, 1 divided by 8** over five successive observations.

In [ ]:
beer = (aus_production.query("ds >= '1992-01-01'")[['ds', 'Beer']]
        .copy().sort_values('ds').reset_index(drop=True))  # A continuous section of the quarterly book series.
beer['ma_4'] = beer['Beer'].rolling(4, center=True).mean()  # Intermediate 4-quarter means.
beer['ma_2x4'] = beer['ma_4'].rolling(2).mean().shift(-1)  # Center the even-order MA on actual quarters.
display(beer[['ds', 'Beer', 'ma_4', 'ma_2x4']].head(8))

plt.plot(beer['ds'], beer['Beer'], label='Observed Beer production')
plt.plot(beer['ds'], beer['ma_2x4'], label='2×4-MA trend')
plt.title('Australian Beer production: 2×4 centered MA')
plt.xlabel('Quarter'); plt.ylabel('Beer production (source units)'); plt.legend()
plt.tight_layout(); plt.show()

**Read the output:** The raw quarterly line contains a recurring within-year pattern. The centered 2×4-MA smooths that four-quarter season while preserving the broad direction. Notice the trend starts and ends a little inside the sample because complete centered windows are required.

## 12 · Monthly employment: the same idea gives a 2×12-MA
In monthly data with annual seasonality, the seasonal length is **12**, also even. Replace 4 with 12 and apply the same two-stage centering. The final average spans 13 months, with half-weight at the two outer months.

In [ ]:
employment_ma = employment[['ds', 'y']].copy()
employment_ma['ma_12'] = employment_ma['y'].rolling(12, center=True).mean()  # Intermediate 12-month mean.
employment_ma['ma_2x12'] = employment_ma['ma_12'].rolling(2).mean().shift(-1)  # Align on a calendar month.
plt.plot(employment_ma['ds'], employment_ma['y'], label='Observed employment')
plt.plot(employment_ma['ds'], employment_ma['ma_2x12'], label='2×12-MA trend-cycle')
plt.title('US Retail Trade: 2×12-MA')
plt.xlabel('Month'); plt.ylabel('Employees (thousands)'); plt.legend()
plt.tight_layout(); plt.show()

**Read the output:** The estimate removes much of the regular annual variation. It is a relatively simple, **fixed-window** trend estimate. STL takes a more flexible smoothing approach and can allow seasonal effects to change over time.

---
# Part E · Classical decomposition
**Classical decomposition** estimates a trend-cycle (using a centered moving average), then a repeating seasonal effect, then the leftover remainder. Compare two ways of combining the pieces:

- **Additive:** `observed = trend + seasonal + remainder`. Seasonal effect is an amount, centered around **0**.
- **Multiplicative:** `observed = trend × seasonal × remainder`. Seasonal effect is a factor, centered around **1**; for example **1.04 means 4% above the trend**.

We use the **same original monthly US Retail Trade series** for both models.

## 13 · Classical additive decomposition
`set_index('ds')` makes the month the Series index, so estimates align to calendar dates. `seasonal_decompose(..., model='additive', period=12)` uses a repeating 12-month seasonal pattern and computes the historical components. With `period=12`, its centered trend will be missing at some beginning and ending months.

In [ ]:
retail = employment.set_index('ds')['y']                     # Monthly employment with dates as index.
add = seasonal_decompose(retail, model='additive', period=12)  # Classical additive model; yearly seasonal cycle.
fig, ax = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for axis, series, title in zip(ax,
    [add.observed, add.trend, add.seasonal, add.resid],
    ['Observed', 'Trend', 'Additive seasonal amount', 'Remainder']):
    axis.plot(series.index, series.values)
    axis.set_title(title)
ax[-1].set_xlabel('Month')
fig.suptitle('US Retail Trade: classical additive decomposition')
plt.tight_layout(); plt.show()

**Read the output:** Seasonal values are measured in **thousands of workers** above or below the smoothed trend. Classical seasonality repeats the same month-by-month pattern each year; this is a simplification compared with STL. The remainder highlights events and changes not captured by that fixed seasonal pattern.

## 14 · Classical multiplicative decomposition
The same history can instead be modeled through seasonal **ratios**. Multiplicative decomposition requires positive observations; retail employment is positive here. Because factors are relative to the trend, their meaning differs from the additive seasonal *amount*.

In [ ]:
mul = seasonal_decompose(retail, model='multiplicative', period=12)  # Classical ratio-to-trend version.
fig, ax = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for axis, series, title in zip(ax,
    [mul.observed, mul.trend, mul.seasonal, mul.resid],
    ['Observed', 'Trend', 'Multiplicative seasonal factor', 'Remainder factor']):
    axis.plot(series.index, series.values)
    axis.set_title(title)
ax[-1].set_xlabel('Month')
fig.suptitle('US Retail Trade: classical multiplicative decomposition')
plt.tight_layout(); plt.show()

**Read the output:** A seasonal factor of `1.04` means about **4% above the trend** in that month; `0.97` means about **3% below**. The multiplicative remainder is also a ratio. Avoid interpreting a seasonal factor such as `1.04` as 1.04 thousand workers.

## 15 · Seasonal indices and adjustment: see the numbers
Classical decomposition assigns a seasonal index to each calendar month. We extract the twelve months with `.index.month`, group by month, and average the index values. Additive indices are normalized to sum to **0** over a year; multiplicative indices average **1**. To seasonally adjust, **subtract** additive seasonality or **divide** by a multiplicative factor.

In [ ]:
add_month = add.seasonal.groupby(add.seasonal.index.month).mean()  # Average additive effect for each Jan...Dec.
mul_month = mul.seasonal.groupby(mul.seasonal.index.month).mean()  # Average ratio for each Jan...Dec.
indices = pd.DataFrame({'Month': pd.date_range('2000-01-01', periods=12, freq='MS').strftime('%b'),
                        'Additive (thousands)': add_month.to_numpy(),
                        'Multiplicative factor': mul_month.to_numpy()})
indices['Relative to trend (%)'] = 100 * (indices['Multiplicative factor'] - 1)
display(indices.round(3))

adjusted = pd.DataFrame({'Observed': retail,
                         'Additive adjusted': add.observed - add.seasonal,  # Subtract seasonal amount.
                         'Multiplicative adjusted': mul.observed / mul.seasonal})  # Divide seasonal factor.
display(adjusted.iloc[12:17].round(2))

**Read the output:** Choose one month in the index table and explain both scales out loud: “This month's additive effect is ... thousand workers; the multiplicative factor is ..., which corresponds to ... percent relative to trend.” The two adjusted series may differ because the models make different assumptions.

**Remember the limitations:** Classical centered MAs lose endpoints; fixed seasonal indices cannot track gradual seasonal changes; outliers can distort moving averages. STL is often useful when seasonality changes over the sample.

---
# Quick reference · Start a fresh Colab notebook later
Keep these short patterns for assignments. **First run the imports and ZIP/CSV upload cells at the top** whenever you start with a fresh Colab runtime.

| Goal | Python to remember | Meaning |
|---|---|---|
| Open a file | `pd.read_csv(DATA_DIR / 'file.csv')` | Read a CSV table. |
| Open dates | `pd.read_csv(..., parse_dates=['ds'])` | Make `ds` a real date. |
| Choose rows | `.query("unique_id == 'Australia'")` | Only rows matching a condition. |
| Keep independent data | `.copy()` | Avoid changing the full original table. |
| Arrange dates | `.sort_values('ds')` | Put earlier dates first. |
| Calculate a column | `df['new'] = df['a'] / df['b']` | Make a new numerical series. |
| Combine same dates | `.merge(other, on='year')` | Join matching annual records. |
| Compute a centered MA | `.rolling(5, center=True).mean()` | Five-point average centered on each date. |
| Decompose monthly data | `STL(values, period=12).fit()` | Estimate trend, seasonal and remainder. |
| Make a plot | `plt.plot(df['ds'], df['y']); plt.show()` | Plot values in time order. |

### A complete mini-example: reopen data, select Australia and plot a centered MA
This is deliberately self-contained **after the upload/import cells**. In a later assignment, change the filename, selected group, measured column and MA window to match the new question.

In [ ]:
df = pd.read_csv(DATA_DIR / 'global_economy.csv')          # Reopen the original annual economy file.
country = df.query("unique_id == 'Australia'").copy()      # Select the country and make a working copy.
country = country.sort_values('ds')                       # Put annual observations in time order.
country['ma_5'] = country['Exports'].rolling(5, center=True).mean()  # Five-year historical smoother.
plt.plot(country['ds'], country['Exports'], label='Observed')
plt.plot(country['ds'], country['ma_5'], label='Centered 5-MA')
plt.xlabel('Year'); plt.ylabel('Exports (% of GDP)'); plt.legend()
plt.tight_layout(); plt.show()

### Final takeaways
Calendar adjustment changes **per-month** to **per-day**; population adjustment changes **total** to **per-person**; inflation adjustment changes the **price basis**. Transformations change the **mathematical scale**. STL and classical decomposition split the series into **trend, seasonality and remainder**. A moving average smooths history, but a **centered** average depends on observations on both sides of its target date. Always state **which dataset, which subgroup, which time unit, and which vertical-axis unit** you are using.